<a href="https://colab.research.google.com/github/Kaique-Passos/FireDetection/blob/main/Treinamento_EfficientNetB0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ===== Parâmetros do treinamento: mude aqui =====

# modelo a treinar (chave de utils/models.py)
# 'efficientnetb0_imagenet': EfficientNet-B0 com pesos pré-treinados na ImageNet (transfer learning)
base_model = 'efficientnetb0_imagenet'

# configuração do K-fold: mantenha igual em todos os notebooks de treino
k_folds = 10
epochs = 100 # máximo de epochs por fold
batch_size = 32
seed_val = 666
dataset_name = 'firenet'
test_dataset_name = 'firenet_test' # None para não avaliar no conjunto de teste
preload_data = True # carrega as imagens na memória
save_models = True # salva os pesos do melhor epoch de cada fold

# early stopping: o fold para quando a validação passa `patience` epochs seguidas
# sem melhorar (nem a acurácia, nem a loss). None treina sempre todos os epochs.
patience = 10

Clonar repositorio

In [ ]:
import os

if os.path.isdir(os.path.join("/content/FireDetection", ".git")):
  %cd /content/FireDetection
  !git pull
  print("Repositório atualizado")
else:
 !git clone https://github.com/Kaique-Passos/FireDetection.git
 print("Repositório clonado")


In [ ]:
# Google Drive: autorize aqui no início, os resultados do treino são salvos nele
from google.colab import drive

if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

history_root = '/content/drive/MyDrive/historico_treinamentos'
os.makedirs(history_root, exist_ok=True)
print('Histórico de treinamentos em:', history_root)

Dependencies install

In [ ]:
!apt-get -qq install imagemagick # required for image transform
!pip install git+https://github.com/angel-ayala/octconv.git # octave convolution library
!pip install thop # model profiler

In [ ]:
# work on repository folder
import os
os.chdir('/content/FireDetection')  #change dir

Obtener datasets

In [ ]:
#firenet
#os.chdir('/content/FireDetection/datasets/FireNetDataset')  #change dir
#!./download.sh

### Solução alternativa: Usando `gdown` (Python)

In [ ]:
# Instale a biblioteca gdown (se ainda não estiver instalada)
!pip install gdown

In [ ]:
import gdown

import shutil
from PIL import Image

# --- Configure o diretório de trabalho se for necessário ---
# Certifique-se de estar no diretório correto para salvar o dataset
# Por exemplo, para o FireNetDataset:
os.chdir('/content/FireDetection/datasets/FireNetDataset')

# arquivo training.zip do meu google drive
FILE_ID = '1pd_i5gCslQJ7KfXxhgGKbdzrZJ_NEAhU'
OUTPUT_FILENAME = 'training.zip'

#Limpando downloads anteriores
pastas = [
    "Test_Dataset1__Our_Own_Dataset",
    "Training",
    "Training Dataset",
    "Test"
]

for pasta in pastas:
    if os.path.exists(pasta):
        shutil.rmtree(pasta)
        print(f'Pasta "{pasta}" deletada.')
    else:
        print(f'Pasta "{pasta}" não existe.')



print(f"Tentando baixar o arquivo com ID '{FILE_ID}' para '{OUTPUT_FILENAME}'...")

try:
    # O gdown.download automaticamente lida com a página de aviso de arquivos grandes
    gdown.download(id=FILE_ID, output=OUTPUT_FILENAME, quiet=False, fuzzy=True)
    print(f"Download de '{OUTPUT_FILENAME}' concluído!")

    # Tentar descompactar e mover (adapte conforme seu download.sh original)
    if os.path.exists(OUTPUT_FILENAME):
        print("Descompactando...")
        !unzip -qq -o {OUTPUT_FILENAME} # -o para sobrescrever, -d para especificar o destino
        # Exemplo de como seu download.sh original move:
        !mv 'Training Dataset' "Training"

        print("Limpando arquivo zip...")
        os.remove(OUTPUT_FILENAME)
    else:
        print(f"Erro: Arquivo '{OUTPUT_FILENAME}' não encontrado após o download.")

    img = Image.open("Training/Fire/fire1.jpg")
    img.convert("RGB").save("Training/Fire/fire1.jpg")

    img = Image.open("Training/NoFire/nofire630.jpg")
    img.convert("RGB").save("Training/NoFire/nofire630.jpg")

    img = Image.open("Training/NoFire/nofire631.jpg")
    img.convert("RGB").save("Training/NoFire/nofire631.jpg")

    img = Image.open("Training/NoFire/nofire650.jpg")
    img.convert("RGB").save("Training/NoFire/nofire650.jpg")

    img = Image.open("Training/NoFire/nf1238.jpg")
    img.convert("RGB").save("Training/NoFire/nf1238.jpg")

except Exception as e:
    print(f"Ocorreu um erro durante o download: {e}")


# Arquivo testing.zip do meu google drive
FILE_ID = '1XyGMfhNQ95EHWp6owMTUN32zj48t8fl8'
OUTPUT_FILENAME = 'testing.zip'

print(f"Tentando baixar o arquivo com ID '{FILE_ID}' para '{OUTPUT_FILENAME}'...")

try:
    # O gdown.download automaticamente lida com a página de aviso de arquivos grandes
    gdown.download(id=FILE_ID, output=OUTPUT_FILENAME, quiet=False, fuzzy=True)
    print(f"Download de '{OUTPUT_FILENAME}' concluído!")

    # Tentar descompactar e mover (adapte conforme seu download.sh original)
    if os.path.exists(OUTPUT_FILENAME):
        print("Descompactando...")
        !unzip -qq -o {OUTPUT_FILENAME} # -o para sobrescrever, -d para especificar o destino
        # Exemplo de como seu download.sh original move:
        !mv 'Test_Dataset1__Our_Own_Dataset' "Test"

        print("Limpando arquivo zip...")
        os.remove(OUTPUT_FILENAME)
    else:
        print(f"Erro: Arquivo '{OUTPUT_FILENAME}' não encontrado após o download.")

except Exception as e:
    print(f"Ocorreu um erro durante o download: {e}")


# Após o download, você pode voltar ao diretório original se necessário
# os.chdir('/content/FireDetection')

In [ ]:
root = '/content/FireDetection/datasets/FireNetDataset'

for item in os.listdir(root):
    print(item)

In [ ]:
for root, dirs, files in os.walk('/content/FireDetection/datasets/FireNetDataset'):
    print(root, '->', len(files), 'arquivos')

In [ ]:
import pandas as pd

path = '/content/FireDetection/datasets/FireNetDataset/dataset.csv'

df = pd.read_csv(path)

print('Total:', len(df))

print('\nPurpose:')
print(df['purpose'].value_counts())

print('\nClasses:')
print(df['class'].value_counts())

print('\nClasses por purpose:')
print(pd.crosstab(df['purpose'], df['class']))

In [ ]:
root = '/content/FireDetection/datasets/FireNetDataset'

df = pd.read_csv(os.path.join(root, 'dataset.csv'))

missing = []

for _, row in df.iterrows():
    path = os.path.join(
        root,
        row['folder_path'],
        row['image_id']
    )

    if not os.path.exists(path):
        missing.append(path)

print('Total no CSV:', len(df))
print('Arquivos encontrados:', len(df) - len(missing))
print('Arquivos ausentes:', len(missing))

if missing:
    print('\nPrimeiros arquivos ausentes:')
    for path in missing[:20]:
        print(path)

In [ ]:
#fismo
#os.chdir('/content/FireDetection/datasets/FiSmoDataset')  #change dir
#!./download.sh

#Limpando downloads anteriores
#pastas = [
#    "FiSmo-Images"
#]

#for pasta in pastas:
#    if os.path.exists(pasta):
#        shutil.rmtree(pasta)
#        print(f'Pasta "{pasta}" deletada.')
#    else:
#        print(f'Pasta "{pasta}" não existe.')


# Arquivo Fismo-images.zip do meu google drive
#FILE_ID = '1WJsbp6ceTaMwtB9UQATy_n6I23Gr0rp7'
#OUTPUT_FILENAME = 'FiSmo-Images.zip'

#print(f"Tentando baixar o arquivo com ID '{FILE_ID}' para '{OUTPUT_FILENAME}'...")

#try:
    # O gdown.download automaticamente lida com a página de aviso de arquivos grandes
#    gdown.download(id=FILE_ID, output=OUTPUT_FILENAME, quiet=False, fuzzy=True)
#    print(f"Download de '{OUTPUT_FILENAME}' concluído!")

    # Tentar descompactar e mover (adapte conforme seu download.sh original)
#    if os.path.exists(OUTPUT_FILENAME):
#        print("Descompactando...")
#        !unzip -qq -o {OUTPUT_FILENAME} # -o para sobrescrever, -d para especificar o destino

        # Tira as imagens de dentro da pasta FiSmo-Images
#        origem = "FiSmo-Images"
#        for item in os.listdir(origem):
#            shutil.move(os.path.join(origem, item), "./")

#        print("Limpando arquivo zip...")
#        os.remove(OUTPUT_FILENAME)
#    else:
#        print(f"Erro: Arquivo '{OUTPUT_FILENAME}' não encontrado após o download.")

#except Exception as e:
#    print(f"Ocorreu um erro durante o download: {e}")

In [ ]:
# back to home
os.chdir('/content/FireDetection')  #change dir

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
import torch
import importlib
from utils.models import get_config

# Carrega a configuração do modelo escolhido em base_model (célula do topo)
config = get_config(base_model)

# Importa a classe do modelo
module = importlib.import_module(config['module_name'])
fire_model = getattr(module, config['class_name'])

# Cria a rede para duas classes (com pesos da ImageNet, baixa os pesos na primeira vez)
model = fire_model(classes=2)

# Simula um lote de 2 imagens RGB no tamanho de entrada do modelo
altura, largura = config['img_dims']
x = torch.randn(2, 3, altura, largura)

# Passa as imagens pela rede
y = model(x)

print('Modelo:', config['model_name'])
print('Parâmetros:', sum(p.numel() for p in model.parameters()))
print('Formato da entrada:', x.shape)
print('Formato da saída:', y.shape)

## Treinamento com K-fold (10 folds)

Junta todas as imagens do FireNet (as linhas `train` e `val` do CSV, 2425 imagens) e divide em 10 folds estratificados, mantendo a proporção fire/no_fire em cada fold. Em cada rodada, 9 folds treinam e 1 fold valida. O melhor epoch (pela acurácia de validação) é avaliado no fold de validação e no conjunto de teste independente (`firenet_test`).

Os resultados ficam no Google Drive, em `MyDrive/historico_treinamentos/<modelo>/`:
- `_em_andamento/`: o treino que está rodando. Se o Colab desconectar, rode as células de novo com a mesma configuração: os folds já concluídos são pulados.
- `AAAA-MM-DD_HH-MM-SS/`: cada treino concluído com sucesso, com a data e a hora (de Brasília) em que terminou.

Cada treino guarda apenas:
- `config.json`: a configuração usada, com início e fim.
- `folds.csv`: em qual fold de validação cada imagem ficou. É a prova de que todas as redes usaram os mesmos folds.
- `test_images.csv`: as 871 imagens do conjunto de teste independente (FireNet Test), as mesmas em todos os folds e em todas as redes.
- `kfold_results.csv` e `kfold_summary.csv`: as métricas de cada fold, a média e o desvio padrão.
- `fold_XX/`: as métricas com a matriz de confusão (`metrics.json`), o histórico por epoch (`history.csv`), as predições de cada imagem no fold de validação e no teste (`predictions_val.csv` e `predictions_test.csv`), os logs do fold e os pesos do melhor epoch (`model_<modelo>.pth`, quando `save_models = True`).

Os parâmetros do treino (`base_model`, `k_folds`, `epochs`, `batch_size`, ...) ficam na primeira célula de código do notebook. Para treinar outra rede, faça uma cópia deste notebook e mude só `base_model` nessa célula. A comparação entre as redes fica no notebook `Comparacao_Modelos.ipynb`.

In [ ]:
import os
import torch
from utils.kfold import run_kfold, start_run, finish_run

# os parâmetros (base_model, k_folds, epochs, ...) ficam na célula do topo do notebook
print('GPU disponível:', torch.cuda.is_available())

In [ ]:
print('Modelo:', base_model)

run_config = {
    'model': base_model,
    'dataset': dataset_name,
    'test_dataset': test_dataset_name,
    'k_folds': k_folds,
    'epochs': epochs,
    'batch_size': batch_size,
    'seed': seed_val,
    'patience': patience
}
run_path = start_run(history_root, base_model, run_config)

resultados, resumo = run_kfold(
    base_model, run_path, dataset_name=dataset_name, k=k_folds, epochs=epochs,
    batch_size=batch_size, seed=seed_val, test_dataset_name=test_dataset_name,
    preload_data=preload_data, save_models=save_models, patience=patience)

# só chega aqui se todos os folds terminaram: move para a pasta com data e hora
run_path = finish_run(history_root, base_model)

display(resultados)
display(resumo)